In [ ]:
from keras.layers import GRU
from keras.layers import Input, Dense
from keras.models import Model
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
from keras.optimizers import Adam
import tensorflow as tf
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from project_brain_decoder.config import get_project_root
from project_brain_decoder.io.nwb_loader import load_nwb
from sklearn.metrics import r2_score

from project_brain_decoder.train_transformer import n_train_steps, n_train_windows

tf.random.set_seed(42)
np.random.seed(42)

In [ ]:
batch_size, window_size, input_dim = 128, 30, 192

In [ ]:
folder = get_project_root() / "data" / "raw"
files = list(folder.glob("*.nwb"))

In [ ]:
train = files[:187] # 60%
val = files[187:249] # 20%
test = files[249:] # 20%

In [ ]:
np.random.shuffle(files)

In [ ]:
def make_windows(neural: np.array, # shape(T, C) - time * channels
                 targets: np.array, # shape(T,) or (T, out_dim)
                 window_size: int,
                 stride: int=10) -> tuple[np.array, np.array]:
    """Slice into (window_size, C) windows; targets aligned to last timestep of each window"""
    T, C = neural.shape
    X = np.lib.stride_tricks.sliding_window_view(neural, window_size, axis=0)[::stride] # (n_windows, C, window_size)
    X = X.transpose(0, 2, 1) # (n_windows, window_size, C)
    # target for each window = value at the end of the window
    y = targets[window_size - 1 :: stride][:X.shape[0]]
    return X, y

In [ ]:
neural_scaler = StandardScaler()
targets_scaler = StandardScaler()

In [ ]:
for file in train:
    session = load_nwb(file)
    neural = np.concatenate([session["neural_spiking_band"], session["neural_threshold_crossings"]])
    targets = np.column_stack([session["target_index_velocity"], session["target_mrs_velocity"]])
    neural_scaler.partial_fit(neural)
    targets_scaler.partial_fit(targets)

In [ ]:
n_train_windows = 0
for file in train:
    session = load_nwb(file)
    neural = session["neural_spiking_band"]
    T = neural.shape[0]
    n_train_windows += T - window_size + 10
n_train_steps = n_train_windows // batch_size
print(n_train_steps)

In [ ]:
n_val_windows = 0
for file in val:
    session = load_nwb(file)
    neural = session["neural_spiking_band"]
    T = neural.shape[0]
    n_val_windows += T - window_size + 10
n_val_steps = n_val_windows // batch_size
print(n_val_steps)

In [ ]:
n_test_windows = 0
for file in test:
    session = load_nwb(file)
    neural = session["neural_spiking_band"]
    T = neural.shape[0]
    n_test_windows += T - window_size + 10
n_test_steps = n_test_windows // batch_size
print(n_test_steps)

In [ ]:
def make_dataset(files, batch_size=batch_size, shuffle=True):
    def generator():
        for file in files:
            session = load_nwb(file)
            neural = np.concatenate([session["neural_spiking_band"], session["neural_threshold_crossings"]], axis=1)
            targets = np.column_stack([session["target_index_velocity"], session["target_mrs_velocity"]])
            X_scaled = neural_scaler.transform(neural)
            y_scaled = targets_scaler.transform(targets)
            X_w, y_w = make_windows(X_scaled, y_scaled, window_size)
            for i in range(len(X_w)):
                yield X_w[i], y_w[i]

        ds = tf.data.Dataset.from_generator(generator=generator,
                                            output_signature=(tf.TensorSpec(shape=(window_size, input_dim), dtype=tf.float32),
                                                              tf.TensorSpec(shape=(2,), dtype=tf.float32)))
        if shuffle:
            ds = ds.shuffle(buffer_size=10_000)
            ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
            return ds

train_ds = make_dataset(train, shuffle=True).repeat()
val_ds = make_dataset(val).repeat()
test_ds = make_dataset(test).repeat()

In [ ]:
def get_gru(window_size, input_dim, GRU):
    input_layer = Input(shape=(window_size, input_dim))
    output_layer = GRU(units=64, dropout=0.3, return_sequences=True)(input_layer)
    output_layer = GRU(units=32, dropout=0.5)(output_layer)
    output_layer = Dense(2)(output_layer)
    gru = Model(inputs=[input_layer], outputs=[output_layer])
    gru.compile(optimizer=Adam(learning_rate=0.0005), loss='mse')
    return gru

In [ ]:
model = get_gru(window_size, input_dim)
model.fit(train_ds, epochs=1,
          steps_per_epoch=n_train_steps,
          validation_data=val_ds,
          validation_steps=n_val_steps,
          callbacks=[EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
                     ReduceLROnPlateau(monitor="val_loss", patience=3)])